# A5 Pandas Analysis: Seattle Crime Data

For this assignment, I am analyzing the Seattle Police Department Crime Data 2008–Present dataset for my Mini Project 1.

Dataset source: https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5/about_data

The CSV file is not included in GitHub because it is larger than GitHub's file size limit. The script/notebook expects the CSV file to be stored locally in the same folder as this notebook.

## Analytical Questions

1. How do reported crime counts vary by hour of day, day of week, and month in Seattle?
2. Which Seattle precincts, sectors, or neighborhoods have the highest number of reported offenses?
3. How do the most common offense categories differ across time periods, such as morning, afternoon, evening, and night?

In [1]:
import pandas as pd

## Load the Dataset

This step loads the Seattle crime CSV into pandas so I can inspect the dataset and answer my three analytical questions.

In [2]:
# This loads the Seattle crime CSV into pandas so I can work with it as a table.
# Once the data is in a DataFrame, I can inspect its structure and summarize patterns.
# The CSV file should be stored in the same folder as this notebook.

df = pd.read_csv(
    "SPD_Crime_Data__2008-Present_20260501.csv",
    low_memory=False
)

## Data Profile

Before answering the analytical questions, I need to understand the size, structure, columns, and missing values in the dataset.

In [3]:
# I first check the dataset size because I want to know the scale of the analysis.
# The result tells me how many offense records and columns are included.

df.shape

(1531135, 20)

In [4]:
# Looking at the first few rows helps me understand what a single record looks like.
# This also gives me a quick check that the file loaded correctly.

df.head()

,Report Number,Report DateTime,Offense ID,Offense Date,NIBRS Group AB,NIBRS Crime Against Category,Offense Sub Category,Shooting Type Group,Block Address,Latitude,Longitude,Beat,Precinct,Sector,Neighborhood,Reporting Area,Offense Category,NIBRS Offense Code Description,NIBRS_offense_code,Census Block 2020
0,2016-091229,2016 Mar 15 02:56:00 PM,7647511654,2016 Mar 15 02:21:00 PM,B,ANY,ALL OTHER,-,93XX BLOCK OF AURORA AVE N,47.696722,-122.344595,N3,North,N,-,704,ALL OTHER,All Other Offenses,90Z,-
1,2020-152553,2020 May 08 07:26:09 PM,13117279109,2020 May 08 11:40:00 AM,A,PROPERTY,BURGLARY,-,58XX BLOCK OF 5TH AVE NE,47.67117473,-122.32282696274,B3,North,B,WALLINGFORD,1545,PROPERTY CRIME,Burglary/Breaking & Entering,220,4500.2007
2,2026-903462,2026 Feb 22 09:01:19 AM,68657554298,2026 Feb 19 01:00:00 PM,A,PROPERTY,LARCENY-THEFT,-,92XX BLOCK OF 35TH AVE SW,47.52017448,-122.376792267112,F2,Southwest,F,ROXHILL/WESTWOOD/ARBOR HEIGHTS,4930,PROPERTY CRIME,All Other Larceny,23H,11402.3008
3,2015-361048,2015 Oct 15 02:05:00 PM,7693699654,2015 Oct 15 02:05:00 PM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,22XX BLOCK OF E MADISON ST,47.61879546,-122.303098812171,C2,East,C,-,5459,ALL OTHER,False Pretenses/Swindle/Confidence Game,26A,-
4,2016-437709,2016 Dec 05 07:19:00 PM,7695857797,2016 Dec 02 10:00:00 AM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,23XX BLOCK OF FRANKLIN AVE E,47.64087835,-122.324662544481,D3,West,D,-,5107,ALL OTHER,Credit Card/Automated Teller Machine Fraud,26B,-


In [5]:
# This gives a more detailed overview of the columns, data types, and non-null counts.
# I use this to confirm that the dataset has the time, offense, and location fields needed for my questions.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531135 entries, 0 to 1531134
Data columns (total 20 columns):
 #   Column                          Non-Null Count    Dtype 
---  ------                          --------------    ----- 
 0   Report Number                   1531135 non-null  object
 1   Report DateTime                 1531135 non-null  object
 2   Offense ID                      1531135 non-null  int64 
 3   Offense Date                    1531135 non-null  object
 4   NIBRS Group AB                  1531135 non-null  object
 5   NIBRS Crime Against Category    1531135 non-null  object
 6   Offense Sub Category            1531135 non-null  object
 7   Shooting Type Group             1531135 non-null  object
 8   Block Address                   1531135 non-null  object
 9   Latitude                        1531135 non-null  object
 10  Longitude                       1531135 non-null  object
 11  Beat                            1531135 non-null  object
 12  Precinct      

In [9]:
# I print the column names so I can use the exact field names in the rest of the notebook.
# This helps prevent mistakes caused by guessing or using column names from a different version of the dataset.

df.columns

Index(['Report Number', 'Report DateTime', 'Offense ID', 'Offense Date',
       'NIBRS Group AB', 'NIBRS Crime Against Category',
       'Offense Sub Category', 'Shooting Type Group', 'Block Address',
       'Latitude', 'Longitude', 'Beat', 'Precinct', 'Sector', 'Neighborhood',
       'Reporting Area', 'Offense Category', 'NIBRS Offense Code Description',
       'NIBRS_offense_code', 'Census Block 2020'],
      dtype='object')

In [10]:
# This missing-value check shows whether any columns have blank values recognized by pandas.
# It helps me decide which fields may need extra caution before drawing conclusions.

df.isnull().sum()

Report Number                     0
Report DateTime                   0
Offense ID                        0
Offense Date                      0
NIBRS Group AB                    0
NIBRS Crime Against Category      0
Offense Sub Category              0
Shooting Type Group               0
Block Address                     0
Latitude                          0
Longitude                         0
Beat                              0
Precinct                          0
Sector                            0
Neighborhood                      0
Reporting Area                    0
Offense Category                  0
NIBRS Offense Code Description    0
NIBRS_offense_code                0
Census Block 2020                 1
dtype: int64

In [11]:
# This gives a first summary of the main offense categories.
# It shows which broad types of offenses make up most of the dataset.

df["Offense Category"].value_counts().head(10)

Offense Category
ALL OTHER         750395
PROPERTY CRIME    700037
VIOLENT CRIME      80703
Name: count, dtype: int64

The dataset has over 1.5 million offense records and includes time, category, and location fields that can answer my three questions. The missing-value check shows very few pandas-recognized missing values, but some location fields use placeholder values such as `-`, which still need caution when interpreting results.

## Prepare Time Columns

To analyze time patterns, I convert `Offense Date` into a datetime field and create separate columns for hour, day of week, month, and year.

In [12]:
# The offense date is stored as text, so I convert it into a datetime value.
# This step makes it possible to analyze crime records by hour, weekday, month, and year.
# I specify the format because the dates in this CSV look like "2016 Mar 15 02:21:00 PM".

df["Offense Date"] = pd.to_datetime(
    df["Offense Date"],
    format="%Y %b %d %I:%M:%S %p",
    errors="coerce"
)

In [13]:
# These new columns break the full date into smaller time units.
# They let me compare reported offenses across hours, weekdays, months, and years.

df["Hour"] = df["Offense Date"].dt.hour
df["Day of Week"] = df["Offense Date"].dt.day_name()
df["Month"] = df["Offense Date"].dt.month_name()
df["Year"] = df["Offense Date"].dt.year

In [14]:
# I preview the new time columns to make sure the transformation worked.
# If these values look correct, I can use them for the time-based analysis.

df[["Offense Date", "Hour", "Day of Week", "Month", "Year"]].head()

,Offense Date,Hour,Day of Week,Month,Year
0,2016-03-15 14:21:00,14,Tuesday,March,2016
1,2020-05-08 11:40:00,11,Friday,May,2020
2,2026-02-19 13:00:00,13,Thursday,February,2026
3,2015-10-15 14:05:00,14,Thursday,October,2015
4,2016-12-02 10:00:00,10,Friday,December,2016


## Question 1: How do reported crime counts vary by hour of day, day of week, and month?

This section summarizes reported offense counts by hour, weekday, month, and year.

In [15]:
# This count shows how reported offenses are distributed across the 24 hours of the day.
# It helps identify whether reports cluster around certain times.

hour_counts = df["Hour"].value_counts().sort_index()
hour_counts

Hour
0     132476
1      53748
2      44049
3      32077
4      28485
5      25323
6      28072
7      35606
8      51157
9      51357
10     55401
11     58137
12     88683
13     67755
14     69314
15     76362
16     77067
17     84173
18     85098
19     77267
20     83102
21     78919
22     78748
23     68759
Name: count, dtype: int64

In [16]:
# This count compares reported offenses across weekdays.
# It helps show whether some days have more reported activity than others.

weekday_counts = df["Day of Week"].value_counts()
weekday_counts

Day of Week
Friday       235243
Saturday     219589
Wednesday    218855
Thursday     218631
Monday       217076
Tuesday      217045
Sunday       204696
Name: count, dtype: int64

In [17]:
# This count summarizes reported offenses by month.
# It gives an initial view of whether reporting volume changes across the year.

month_counts = df["Month"].value_counts()
month_counts

Month
May          134177
January      133671
October      131680
July         131580
August       131226
March        130242
September    127214
April        126768
June         123463
December     123068
November     122161
February     115885
Name: count, dtype: int64

In [18]:
# I also check year counts because this dataset covers a long time range.
# This helps me see whether the records are evenly distributed over time or concentrated in certain years.

year_counts = df["Year"].value_counts().sort_index()
year_counts

Year
1900        6
1908        1
1915        1
1920        1
1929        1
        ...  
2022    87729
2023    83100
2024    82197
2025    77646
2026    23096
Name: count, Length: 67, dtype: int64

The time-based summaries show that reported offenses are not evenly distributed across time. Hour 0 has the highest count, Friday has the highest weekday count, and May has the highest monthly count. These are raw offense counts, so they show reporting volume rather than proving exact crime risk.

## Question 2: Which Seattle precincts, sectors, or neighborhoods have the highest number of reported offenses?

This section summarizes reported offense counts by precinct, sector, and neighborhood.

In [19]:
# This summarizes reported offenses by precinct, which is a broad geographic category.
# It shows where reports are most concentrated at the police precinct level.

precinct_counts = df["Precinct"].value_counts(dropna=False)
precinct_counts

Precinct
North        476605
West         412686
East         250215
South        225023
Southwest    151808
-              9461
OOJ            5337
Name: count, dtype: int64

In [20]:
# Sector is more detailed than precinct, so this gives a more specific location breakdown.
# The top sectors help identify smaller areas with high numbers of offense records.

sector_counts = df["Sector"].value_counts(dropna=False).head(15)
sector_counts

Sector
U    117864
E    112569
K    110430
B    107545
D    103896
M    103469
R     97396
Q     94874
L     89013
N     87612
S     78575
W     77438
J     74552
F     74360
C     68905
Name: count, dtype: int64

In [51]:
# This looks at neighborhood labels to see which named areas appear most often.
# I need to interpret this carefully because many records use "-" instead of a real neighborhood name.

neighborhood_counts = df["Neighborhood"].value_counts(dropna=False).head(15)
neighborhood_counts

Neighborhood
-                                   744492
DOWNTOWN COMMERCIAL                  55904
CAPITOL HILL                         52408
NORTHGATE                            43786
QUEEN ANNE                           40960
SLU/CASCADE                          36025
UNIVERSITY                           31834
ROOSEVELT/RAVENNA                    29103
BALLARD SOUTH                        28384
FIRST HILL                           27621
CHINATOWN/INTERNATIONAL DISTRICT     24571
LAKECITY                             21006
CENTRAL AREA/SQUIRE PARK             20372
BELLTOWN                             19770
SANDPOINT                            17101
Name: count, dtype: int64

In [52]:
# This is a groupby version of the precinct count.
# It confirms the precinct-level pattern by counting offense IDs within each precinct.

precinct_grouped = df.groupby("Precinct")["Offense ID"].count().sort_values(ascending=False)
precinct_grouped

Precinct
North        476605
West         412686
East         250215
South        225023
Southwest    151808
-              9461
OOJ            5337
Name: Offense ID, dtype: int64

In [41]:
# This combines location and offense type in one summary.
# It helps show which offense categories are most common within the highest-volume neighborhood labels.

neighborhood_offense_counts = (
    df.groupby(["Neighborhood", "Offense Category"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

neighborhood_offense_counts

Neighborhood                      Offense Category
-                                 ALL OTHER           384294
                                  PROPERTY CRIME      326889
                                  VIOLENT CRIME        33309
DOWNTOWN COMMERCIAL               ALL OTHER            31048
CAPITOL HILL                      ALL OTHER            27750
QUEEN ANNE                        PROPERTY CRIME       23745
CAPITOL HILL                      PROPERTY CRIME       20965
NORTHGATE                         PROPERTY CRIME       20587
                                  ALL OTHER            20569
DOWNTOWN COMMERCIAL               PROPERTY CRIME       20492
ROOSEVELT/RAVENNA                 PROPERTY CRIME       18054
SLU/CASCADE                       ALL OTHER            17549
UNIVERSITY                        PROPERTY CRIME       17191
SLU/CASCADE                       PROPERTY CRIME       16118
BALLARD SOUTH                     PROPERTY CRIME       15918
QUEEN ANNE                        

The location summaries show that North and West precincts have the highest reported offense counts. At the neighborhood level, many records are labeled `-`, so neighborhood-level findings need caution. Among named neighborhoods, Downtown Commercial and Capitol Hill appear near the top.

## Question 3: How do the most common offense categories differ across time periods?

This section creates time periods from the hour column and compares offense categories across morning, afternoon, evening, and night.

In [42]:
# This function turns each hour into a broader part of the day.
# Grouping records this way makes it easier to compare morning, afternoon, evening, and night.

def assign_time_period(hour):
    if pd.isnull(hour):
        return "Unknown"
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

In [43]:
# I apply the time period labels to every row.
# The new column gives me a simpler way to compare offense patterns across the day.

df["Time Period"] = df["Hour"].apply(assign_time_period)

In [44]:
# This checks how many records fall into each time period.
# It shows whether one part of the day has noticeably more reported offenses than the others.

df["Time Period"].value_counts()

Time Period
Night        517261
Afternoon    379181
Evening      329640
Morning      305053
Name: count, dtype: int64

In [45]:
# This summary compares offense categories within each time period.
# It helps show whether the most common offense types change depending on the time of day.

time_offense_counts = (
    df.groupby(["Time Period", "Offense Category"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

time_offense_counts

Time Period  Offense Category
Night        ALL OTHER           256766
             PROPERTY CRIME      227953
Afternoon    ALL OTHER           196709
Evening      PROPERTY CRIME      168192
Afternoon    PROPERTY CRIME      164772
Morning      ALL OTHER           152482
Evening      ALL OTHER           144438
Morning      PROPERTY CRIME      139120
Night        VIOLENT CRIME        32542
Afternoon    VIOLENT CRIME        17700
Evening      VIOLENT CRIME        17010
Morning      VIOLENT CRIME        13451
dtype: int64

In [46]:
# This table puts time periods and offense categories side by side.
# It is easier to compare category counts across the day in this format.

time_offense_table = (
    df.groupby(["Time Period", "Offense Category"])
    .size()
    .unstack(fill_value=0)
)

time_offense_table

Offense Category,ALL OTHER,PROPERTY CRIME,VIOLENT CRIME
Time Period,,,
Afternoon,196709,164772,17700
Evening,144438,168192,17010
Morning,152482,139120,13451
Night,256766,227953,32542


In [47]:
# Here I focus only on nighttime records.
# This subset lets me see which broad offense categories are most common at night.

night_crimes = df[df["Time Period"] == "Night"]

night_crimes["Offense Category"].value_counts().head(10)

Offense Category
ALL OTHER         256766
PROPERTY CRIME    227953
VIOLENT CRIME      32542
Name: count, dtype: int64

In [48]:
# I also create a morning subset for comparison.
# Comparing morning and night helps show whether category patterns shift across the day.

morning_crimes = df[df["Time Period"] == "Morning"]

morning_crimes["Offense Category"].value_counts().head(10)

Offense Category
ALL OTHER         152482
PROPERTY CRIME    139120
VIOLENT CRIME      13451
Name: count, dtype: int64

Night has the highest count among the four time-period groups. Across both night and morning, the most common broad categories are `ALL OTHER`, `PROPERTY CRIME`, and `VIOLENT CRIME`. This suggests the broad category ranking is similar across time periods, even though the total volume changes.

## Additional Data Check: Reports vs. Offenses

The dataset includes both `Report Number` and `Offense ID`. Since one report can include multiple offenses, I check whether I am counting offense records or unique police reports.

In [49]:
# I compare unique report numbers with unique offense IDs because they are not the same thing.
# This tells me whether my analysis is counting offense records or unique police reports.

unique_reports = df["Report Number"].nunique()
unique_offenses = df["Offense ID"].nunique()

unique_reports, unique_offenses

(1349939, 1531134)

In [50]:
# This shows whether some report numbers appear many times.
# If they do, it means one police report can contain multiple offenses, which affects how raw counts should be interpreted.

df["Report Number"].value_counts().head(10)

Report Number
2025-162098    18
2023-018027    18
2023-124881    16
2023-346113    15
2024-337659    14
2020-220030    13
2022-015811    13
2022-297056    12
2023-240907    12
2026-037979    12
Name: count, dtype: int64

There are fewer unique report numbers than unique offense IDs, so this analysis should be described as counting offense records rather than unique police reports. This matters because one police report can include multiple offenses.

## Summary

This notebook gives initial answers to my three Mini Project 1 questions using pandas. I used `head()`, `info()`, `isnull().sum()`, `value_counts()`, filtering, and `groupby()` to inspect the dataset and summarize time, location, and offense category patterns.

These findings should be interpreted as raw offense counts, not direct measures of crime risk. Future analysis should be careful about placeholder location values like `-`, the difference between report numbers and offense IDs, and contextual factors such as population, foot traffic, and reporting behavior.